In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb

In [ ]:
# 1) Load data
def load_data(path):
    df = pd.read_csv(path)
    return df

In [ ]:
# 2) Basic preprocess
def preprocess(df):
    df = df.copy()

    # Drop ID column
    if "id" in df.columns:
        df = df.drop(columns=["id"])

    # Split features / target
    X = df.drop(columns=["loss"])
    y = df["loss"]

    # Log-transform target (VERY IMPORTANT)
    y_log = np.log1p(y)

    return X, y_log

In [ ]:
def get_feature_types(X):
    categorical_cols = [col for col in X.columns if col.startswith("cat")]
    numeric_cols = [col for col in X.columns if col.startswith("cont")]
    return categorical_cols, numeric_cols


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

def build_preprocessor(categorical_cols, numeric_cols):
    preprocessor = ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
            ("num", StandardScaler(), numeric_cols)
        ]
    )
    return preprocessor


In [ ]:
from sklearn.linear_model import Ridge

def build_ridge_model(preprocessor):
    model = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("ridge", Ridge())
    ])
    return model


In [ ]:
def train_model(model, X_train, y_train):
    model.fit(X_train, y_train)
    return model


In [ ]:
def evaluate(model, X_test, y_test):
    preds_log = model.predict(X_test)

    # Convert back to original
    preds = np.expm1(preds_log)
    y_true = np.expm1(y_test)

    rmse = np.sqrt(mean_squared_error(y_true, preds))
    r2 = r2_score(y_true, preds)

    print("RMSE:", rmse)
    print("R²:", r2)


In [ ]:
def main():
    df = load_data("train.csv")

    X, y_log = preprocess(df)
    categorical_cols, numeric_cols = get_feature_types(X)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y_log, test_size=0.2, random_state=42
    )

    preprocessor = build_preprocessor(categorical_cols, numeric_cols)
    ridge_model = build_ridge_model(preprocessor)

    ridge_model = train_model(ridge_model, X_train, y_train)
    evaluate(ridge_model, X_test, y_test)


if __name__ == "__main__":
    main()
